In [2]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv
load_dotenv()
MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]

In [4]:
#re-ranking
vectorstore = FAISS.from_documents(documents, embeddings_model)
bm_25_retriever = BM25Retriever.from_documents(documents, k=5)

In [5]:
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in doc_embeddings:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

In [7]:
#re-ranking 필요한 이유 임베딩 검색 내에서 document에서 유사한 값을 가지오는데
#query와 doc의 상호관계
def vector_search(query, vectorstore, top_k=5):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc, 1.0 / (1.0 + score)) for doc, score in results]

query = "트랜스포머와 BERT의 관계"
results = vector_search(query, vectorstore)
results

[(Document(id='92250bb5-434e-40d2-979e-7954f6d7c695', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.5249158)),
 (Document(id='8d50ba2a-f2b3-4ffa-9310-9c16b986c5f8', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.46390262)),
 (Document(id='94ffd756-aca6-404e-a2c7-e36a6317a320', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41886568)),
 (Document(id='4c5dd121-515c-4488-bf3a-cfe9750279f2', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.4128145)),
 (Document(id='0e78f924-26ff-4d7e-abb4-b228735717d0', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39726147))]

In [8]:
def keyword_rerank(query, search_results):
    query_terms = set(query.lower().split())
    reranked = []

    for doc, orig_score in search_results:
        doc_terms = doc.page_content.lower().split()
        keyword_hits = sum(1 for t in doc_terms if t in query_terms)
        new_score = orig_score + 0.1 * keyword_hits
        reranked.append((doc, new_score, orig_score))

    reranked.sort(key = lambda x:x[1], reverse=True)
    return reranked

In [9]:
reranked = keyword_rerank(query, results)
reranked

[(Document(id='92250bb5-434e-40d2-979e-7954f6d7c695', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.5249158),
  np.float32(0.5249158)),
 (Document(id='8d50ba2a-f2b3-4ffa-9310-9c16b986c5f8', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.46390262),
  np.float32(0.46390262)),
 (Document(id='94ffd756-aca6-404e-a2c7-e36a6317a320', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41886568),
  np.float32(0.41886568)),
 (Document(id='4c5dd121-515c-4488-bf3a-cfe9750279f2', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.4128145),
  np.float32(0.4128145)),
 (Document(id='0e78f924-26ff-4d7e-abb4-b228735717d0', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39726147),
  np.float32(0.3972614

In [10]:
def rank_change(original_results, reranked_results):
    before_rank = {
        doc.metadata['id']: i
        for i, (doc, _) in enumerate(original_results)
    }

    result = []
    for j, (doc, _, _) in enumerate(reranked_results):
        doc_id = doc.metadata['id']
        before = before_rank[doc_id]
        after = j
        change = before - after

        result.append((doc_id, before, after, change))

    return result

In [11]:
rank_changes = rank_change(results, reranked)

for doc_id, before, after, change in rank_changes:
    print(f"{doc_id}: {before} → {after} (change={change})")

d1: 0 → 0 (change=0)
d2: 1 → 1 (change=0)
d3: 2 → 2 (change=0)
d6: 3 → 3 (change=0)
d7: 4 → 4 (change=0)


In [13]:
#ranked -> Cross-encoder / LLM -> reranked
#vector_search 기존에 쿼리와 문서를 따로따로 임베딩처리 : Bi-encoder

In [ ]:
"""
rerank에 대한 3가지 전략
(쿼리 - 문서) -> rerank
LLM -> rerank
BM25 -> rerank
"""

In [18]:
class BM25Reranker:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def rerank(self, query, search_results):
        docs = [doc for doc, _ in search_results]
        tokenized = [doc.page_content.lower().split() for doc in docs]

        avg_dl = np.mean([len(t) for t in tokenized])

        df_count = Counter()
        for tokens in tokenized:
            for t in set(tokens):
                df_count[t] += 1
        N = len(docs)
        
        query_tokens = query.lower().split()
        scored = []

        for i, doc in enumerate(docs):
            score = 0.0
            doc_len = len(tokenized[i])
            tf_counter = Counter(tokenized[i])

            for qt in query_tokens:
                tf = tf_counter.get(qt, 0)
                if tf == 0:
                    continue
                df = df_count.get(qt, 0)
                idf = math.log((N-df + 0.5) / (df + 0.5) + 1)
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * doc_len /avg_dl)
                score +=  idf * numerator / denominator

            scored.append((doc, score))

        scored.sort(key = lambda x: x[1], reverse=True)
        return scored

In [19]:
bm25_reranker = BM25Reranker()
bm25_results = bm25_reranker.rerank(query, results)
bm25_results

[(Document(id='92250bb5-434e-40d2-979e-7954f6d7c695', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.0),
 (Document(id='8d50ba2a-f2b3-4ffa-9310-9c16b986c5f8', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.0),
 (Document(id='94ffd756-aca6-404e-a2c7-e36a6317a320', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.0),
 (Document(id='4c5dd121-515c-4488-bf3a-cfe9750279f2', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.0),
 (Document(id='0e78f924-26ff-4d7e-abb4-b228735717d0', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.0)]

In [37]:
def llm_rerank(query, search_results, top_k=3):
    docs_text = '\n'.join(
        f"[{doc.metadata['id']}] {doc.page_content}" 
        for doc, _ in search_results
    )
    
    score_template = ", ".join(
        f'"{doc.metadata["id"]}": 0.0-1.0'
        for doc, _ in search_results
    )

    scoring_chain = ChatPromptTemplate.from_messages([
        ('system', '당신은 scoring 시스템 입니다. 항상 json 형태로 출력하세요'),
        ('human', """다음 쿼리에 대해 각 문서의 관련성을 0.0 ~ 1.0으로 평가하세요

        쿼리 : {query}

        문서들 :
        {docs_text}

        JSON으로 답하세요:
        {{"scores": {{{score_template}}}}}
        """)
    ]) | llm | StrOutputParser()

    result = scoring_chain.invoke({
        'query': query,
        'docs_text': docs_text,
        'score_template': score_template
    })

    cleaned = result.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned.replace('```json', '')
        cleaned = cleaned.replace('```', '')
    parsed = json.loads(cleaned)
    scores = parsed.get('scores', {})

    scored = []

    for doc, orig in search_results:
        rerank_score = scores.get(doc.metadata['id'], 0.0)
        scored.append((doc, float(rerank_score), orig))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

In [38]:
llm_result = llm_rerank(query, results, top_k=3)
llm_result

[(Document(id='8d50ba2a-f2b3-4ffa-9310-9c16b986c5f8', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  1.0,
  np.float32(0.46390262)),
 (Document(id='94ffd756-aca6-404e-a2c7-e36a6317a320', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.3,
  np.float32(0.41886568)),
 (Document(id='92250bb5-434e-40d2-979e-7954f6d7c695', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.2,
  np.float32(0.5249158))]

In [41]:
def hybrid_rerank(query, search_results, bm25_weight=0.3):
    bm25 = BM25Reranker()
    bm25_scored = bm25.rerank(query, search_results)
    bm25_map = {doc.metadata['id'] : score for doc, score in bm25_scored}

    llm_scored = llm_rerank(query, search_results, top_k=len(search_results))
    llm_map = {doc.metadata['id'] : score for doc, score, _ in llm_scored}

    def normalize(scores_dict):
        vals = list(scores_dict.values())
        min_, max_ = min(vals), max(vals)
        range_ = max_ - min_ if max_ > min_ else 1e-8
        return {k: (v -min_) / range_ for k, v in scores_dict.items()}


    bm25_norm = normalize(bm25_map)
    llm_norm = normalize(llm_map)

    combined = []
    for doc, _ in search_results:
        did = doc.metadata['id']
        score = bm25_weight * bm25_norm.get(did, 0) + (1-bm25_weight) * llm_norm.get(did, 0)

        combined.append((doc, score))
    combined.sort(key=lambda x: x[1], reverse=True)
    return combined

In [42]:
hybrid = hybrid_rerank(query, results)
hybrid

[(Document(id='8d50ba2a-f2b3-4ffa-9310-9c16b986c5f8', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.7),
 (Document(id='94ffd756-aca6-404e-a2c7-e36a6317a320', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.23333333333333334),
 (Document(id='92250bb5-434e-40d2-979e-7954f6d7c695', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.07777777777777778),
 (Document(id='4c5dd121-515c-4488-bf3a-cfe9750279f2', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.0),
 (Document(id='0e78f924-26ff-4d7e-abb4-b228735717d0', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.0)]

In [49]:
def llm_listwise_rerank(query, search_results):
    docs_text = '\n'.join(
        f"{i+1}.[{doc.metadata['id']}] {doc.page_content[:80]}"
        for i, (doc, _) in enumerate(search_results)
    )

    ranking_chain = ChatPromptTemplate.from_messages([
        ('system', '당신은 문서 랭킹 시스템입니다.'),
        ('human', """다음 문서들을 쿼리와의 관련성 순서로 정렬하세요.

        쿼리 : {query}

        문서들 :
        {docs_text}

        가장 관련성 높은 순서대로 문서 번호를 쉼표로 나열하세요 (예: 3, 1, 5, 2, 4):
        """)
    ]) | llm | StrOutputParser()

    result = ranking_chain.invoke({
        'query': query,
        'docs_text': docs_text,
    })

    order = [int(x.strip()) for x in result.strip().split(',')]

    docs_list = [doc for doc, _ in search_results]

    reranked = []
    for rank, idx in enumerate(order):
        if 1 <= idx <= len(docs_list):
            doc = docs_list[idx-1]
            reranked.append((doc, 1.0 - rank * 0.1))

    return reranked

In [50]:
listwise = llm_listwise_rerank(query, results)